In [ ]:
!pip install -q transformers peft bitsandbytes accelerate faiss-cpu rouge-score tqdm pylcs

  Preparing metadata (setup.py) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 41.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 113.6 MB/s eta 0:00:00


In [24]:
# =============================================================================
# BOOTSTRAP: restore full pipeline state from Drive caches (~5-10 min, CPU ok)
# Run this first after ANY runtime restart.
# =============================================================================
import os, re as _re, json, pickle, gc
import numpy as np, pandas as pd, faiss
from pathlib import Path
from tqdm import tqdm
from collections import Counter
from rouge_score import rouge_scorer
from rouge_score import tokenize as rs_tokenize
try:
    import pylcs; HAVE_PYLCS = True
except ImportError:
    os.system('pip install -q pylcs')
    try: import pylcs; HAVE_PYLCS = True
    except Exception: HAVE_PYLCS = False

from google.colab import drive
if not Path('/content/drive/MyDrive').exists(): drive.mount('/content/drive')
BASE = Path('/content/drive/MyDrive/multilingual-health-qa')
DATA_DIR, OUTPUT_DIR = BASE/'data', BASE/'outputs'
CACHE = OUTPUT_DIR/'mbr_cache'

# ---- data ----
train_df = pd.read_csv(DATA_DIR/'Train.csv')
val_df   = pd.read_csv(DATA_DIR/'Val.csv')
test_df  = pd.read_csv(DATA_DIR/'Test.csv')
sample_sub = pd.read_csv(DATA_DIR/'SampleSubmission.csv'); SUB_COLS = list(sample_sub.columns)
FT_MODEL_DIR = OUTPUT_DIR / 'qwen-ft-health'
combined = pd.concat([train_df, val_df], ignore_index=True).dropna(subset=['input','output']).reset_index(drop=True)
questions_raw = combined['input'].astype(str).tolist()
answers_raw   = combined['output'].astype(str).tolist()
subsets_raw   = combined['subset'].astype(str).tolist()
corpus_q_stripped = [q.strip() for q in questions_raw]
val_qs  = val_df['input'].fillna('').astype(str).tolist()
test_qs = test_df['input'].fillna('').astype(str).tolist()
test_subs = test_df['subset'].fillna('').astype(str).tolist()
SUBSET_TO_LANG = {'Aka_Gha':'Akan (Ghana)','Amh_Eth':'Amharic (Ethiopia)','Eng_Eth':'English (Ethiopia)',
 'Eng_Gha':'English (Ghana)','Eng_Ken':'English (Kenya)','Eng_Uga':'English (Uganda)',
 'Lug_Uga':'Luganda (Uganda)','Swa_Ken':'Swahili (Kenya)'}
scorer_both = rouge_scorer.RougeScorer(['rouge1','rougeL'], use_stemmer=False)

# ---- embeddings (all cached) ----
corpus_emb = np.load(CACHE/'emb_corpus.npy'); val_emb = np.load(CACHE/'emb_val.npy'); test_emb = np.load(CACHE/'emb_test.npy')
gem_corpus = np.load(CACHE/'gem_emb_corpus.npy'); gem_val = np.load(CACHE/'gem_emb_val.npy'); gem_test = np.load(CACHE/'gem_emb_test.npy')
ans_emb = np.load(CACHE/'emb_corpus_answers.npy')
bge_corpus = np.load(CACHE/'bge_corpus.npy'); bge_val = np.load(CACHE/'bge_val.npy'); bge_test = np.load(CACHE/'bge_test.npy')

# ---- indices ----
def build_lang_idx(emb):
    out = {}
    for sub in sorted(set(subsets_raw)):
        mask = [i for i,s in enumerate(subsets_raw) if s==sub]
        ix = faiss.IndexFlatIP(emb.shape[1]); ix.add(emb[mask]); out[sub]=(ix,mask)
    return out
lang_indices = build_lang_idx(corpus_emb); gem_lang_idx = build_lang_idx(gem_corpus)
qa_idx = build_lang_idx(ans_emb); bge_idx = build_lang_idx(bge_corpus)
global_idx = faiss.IndexFlatIP(corpus_emb.shape[1]); global_idx.add(corpus_emb)

# ---- pickled state ----
val_cands_all = pickle.load(open(CACHE/'val_cands.pkl','rb'))
val_prep, val_refscores = pickle.load(open(CACHE/'val_prep.pkl','rb'))
v4c, v4p, v4r = pickle.load(open(CACHE/'val_union4.pkl','rb'))
P = pickle.load(open(CACHE/'uni_rebuild.pkl','rb'))
llm_ans = json.load(open(OUTPUT_DIR/'gemini_mbr_llm_prog.json'))

# ---- core functions (canonical definitions) ----
K_CANDIDATES, K_LEG, CAP = 15, 20, 400
_UNI = _re.compile(r'\w+', _re.UNICODE)
def uni_toks(t): return _UNI.findall(t.lower())
def _lcs_py(a,b):
    if not a or not b: return 0
    dp=[0]*(len(b)+1)
    for ai in a:
        prev=0
        for j,bj in enumerate(b):
            cur=dp[j+1]; dp[j+1]=prev+1 if ai==bj else max(dp[j+1],dp[j]); prev=cur
    return dp[-1]
def lcs_tok(a,b):
    if HAVE_PYLCS:
        v={}
        for t in a:
            if t not in v: v[t]=len(v)
        for t in b:
            if t not in v: v[t]=len(v)
        return pylcs.lcs_sequence_length(''.join(chr(0x100+v[t]) for t in a), ''.join(chr(0x100+v[t]) for t in b))
    return _lcs_py(a,b)
def uni_r1(rt,ht):
    if not rt or not ht: return 0.0
    return 2*sum((Counter(rt)&Counter(ht)).values())/(len(rt)+len(ht))
def uni_rl(rt,ht):
    if not rt or not ht: return 0.0
    return 2*lcs_tok(rt,ht)/(len(rt)+len(ht))
def get_same_lang_candidates(q_text,q_emb,subset,k=K_CANDIDATES,exclude_exact=True):
    qs=q_text.strip()
    if subset in lang_indices:
        idx,mask=lang_indices[subset]
        D,I=idx.search(np.asarray(q_emb,np.float32).reshape(1,-1),min(k+5,len(mask)))
        out=[]
        for d,li in zip(D[0],I[0]):
            if li<0: continue
            ci=mask[int(li)]
            if exclude_exact and corpus_q_stripped[ci]==qs: continue
            out.append({'answer':answers_raw[ci],'sim':float(d),'idx':ci})
            if len(out)>=k: break
        if out: return out
    return []
def union4(q_text,afri_q,gem_q,bge_q,subset,exclude_exact=True):
    qs=q_text.strip(); rrf={}
    for idx_map,emb in [(lang_indices,afri_q),(gem_lang_idx,gem_q),(qa_idx,afri_q),(bge_idx,bge_q)]:
        if subset not in idx_map: continue
        idx,mask=idx_map[subset]
        D,I=idx.search(np.asarray(emb,np.float32).reshape(1,-1),min(K_LEG+5,len(mask)))
        r=0
        for li in I[0]:
            if li<0: continue
            ci=mask[int(li)]
            if exclude_exact and corpus_q_stripped[ci]==qs: continue
            rrf[ci]=rrf.get(ci,0.0)+1.0/(60+r); r+=1
            if r>=K_LEG: break
    ranked=sorted(rrf.items(),key=lambda kv:-kv[1])[:35]
    return [{'answer':answers_raw[ci],'sim':float(np.dot(afri_q,corpus_emb[ci])),'idx':ci} for ci,_ in ranked]
def uni_prep(cands,max_tok=80):
    answers=[c['answer'] for c in cands]
    w=np.exp(np.array([c['sim'] for c in cands])*5); w/=w.sum()
    seen,dd,ddw={},[],[]
    for a,wi in zip(answers,w):
        k=a.strip().lower()
        if k in seen: ddw[seen[k]]+=wi
        else: seen[k]=len(dd); dd.append(a); ddw.append(wi)
    ddw=np.array(ddw); ddw/=ddw.sum(); n=len(dd)
    toks=[uni_toks(a)[:max_tok] for a in dd]
    if n==1: return dd,ddw,np.zeros(1),np.zeros(1)
    S1,SL=np.zeros((n,n)),np.zeros((n,n))
    for i in range(n):
        for j in range(i+1,n):
            S1[i,j]=S1[j,i]=uni_r1(toks[i],toks[j]); SL[i,j]=SL[j,i]=uni_rl(toks[i],toks[j])
    return dd,ddw,S1@ddw,SL@ddw
def mbr_idx(ub,ddw,alpha,margin):
    if len(ddw)==1: return 0
    u=ub+alpha*ddw; b=int(np.argmax(u))
    return b if (b!=0 and u[b]-u[0]>margin) else 0
uni_prior={sub: float(np.median([len(uni_toks(answers_raw[i])) for i,s in enumerate(subsets_raw) if s==sub])) for sub in SUBSET_TO_LANG}
def uni_stitch(cands,lam,sub):
    answers=[c['answer'] for c in cands]
    w=np.exp(np.array([c['sim'] for c in cands])*5); w/=w.sum()
    p=Counter()
    for a,wi in zip(answers,w):
        for t,c in Counter(uni_toks(a)[:CAP]).items(): p[t]+=wi*c
    ref_len=max(lam*uni_prior[sub],1.0); seen,pool=set(),[]
    for a in answers:
        for s in _re.split(r'(?<=[.!?])\s+|\n+',a):
            s=s.strip(); st=uni_toks(s)
            if len(st)<4: continue
            k=' '.join(st)
            if k not in seen: seen.add(k); pool.append((s,Counter(st),len(st)))
    if not pool: return answers[0]
    def ef1(cov,hl):
        mt=sum(min(c,p[t]) for t,c in cov.items())
        if hl==0 or mt==0: return 0.0
        pr_,rc=mt/hl,mt/ref_len; return 2*pr_*rc/(pr_+rc)
    chosen,cov,hl,cur=[],Counter(),0,0.0
    while pool:
        bi,bg=-1,0.0
        for i,(s,sc_,sl) in enumerate(pool):
            nc=cov.copy(); nc.update(sc_)
            g=ef1(nc,hl+sl)-cur
            if g>bg: bg,bi=g,i
        if bi<0: break
        s,sc_,sl=pool.pop(bi); chosen.append(s); cov.update(sc_); hl+=sl; cur+=bg
    return ' '.join(chosen) if chosen else answers[0]

# ---- tuned decisions from the unicode rebuild (hardcoded = reproducible) ----
choice = {'Aka_Gha':('2leg',0.10,0.010),'Amh_Eth':('2leg',0.05,99.0),'Eng_Eth':('2leg',0.05,99.0),
          'Eng_Gha':('4leg',0.20,0.050),'Eng_Ken':('2leg',0.05,99.0),'Eng_Uga':('2leg',0.05,99.0),
          'Lug_Uga':('2leg',0.05,99.0),'Swa_Ken':('2leg',0.05,99.0)}
uni_stitch_gate = {'Aka_Gha':{'use':True,'lam':0.70,'pool':'2leg'},
                   'Eng_Gha':{'use':True,'lam':0.85,'pool':'4leg'},
                   'Amh_Eth':{'use':True,'lam':0.70,'pool':'2leg'}}
print("STATE RESTORED:", len(combined), "corpus |", len(llm_ans), "LLM answers | all caches loaded")

STATE RESTORED: 36501 corpus | 2618 LLM answers | all caches loaded


In [ ]:
"""
=============================================================================
COMPLETE NOTEBOOK
=============================================================================
"""
# ===========================================================================
# STEP 1: LOAD FINE-TUNED MODEL (4-BIT — 3.5GB, FAST)
# ===========================================================================
import torch, gc, json, re as _re2, time
import numpy as np
from datetime import datetime
from pathlib import Path
from tqdm import tqdm
from collections import defaultdict
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
def log(msg):
    print(f"[{datetime.now().strftime('%H:%M:%S')}] {msg}", flush=True)
FT_MODEL_DIR = OUTPUT_DIR / 'qwen-ft-health'
assert (FT_MODEL_DIR / 'adapter_config.json').exists(), \
    f"No fine-tuned model at {FT_MODEL_DIR}. Train first."
log("Loading fine-tuned Qwen in 4-bit (fast inference)...")
bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)
tok = AutoTokenizer.from_pretrained(str(FT_MODEL_DIR))
base = AutoModelForCausalLM.from_pretrained(
    'Qwen/Qwen2.5-7B-Instruct',
    quantization_config=bnb,
    device_map='auto',
    trust_remote_code=True,
)
ft = PeftModel.from_pretrained(base, str(FT_MODEL_DIR))
ft.eval()
log(f"Model loaded in 4-bit. GPU mem: {torch.cuda.memory_allocated()/1e9:.1f}GB")
# ===========================================================================
# FAST GENERATION FUNCTIONS (short context, low max_tokens)
# ===========================================================================
@torch.no_grad()
def ft_generate(q, lang, cands, max_new=200):
    # Only top-3 context, truncated to 200 chars each
    ctx = "\n".join(f"{k+1}. {c['answer'][:200]}" for k, c in enumerate(cands[:3]))
    msgs = [
        {"role": "system", "content":
         f"You are a health expert. Answer using the EXACT words from the references. "
         f"Be complete. Answer in {lang}."},
        {"role": "user", "content":
         f"Question: {q}\n\nReferences:\n{ctx}\n\nAnswer in {lang}:"}
    ]
    text = tok.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False)
    inputs = tok(text, return_tensors='pt', truncation=True, max_length=1024).to(ft.device)
    out = ft.generate(
        **inputs, max_new_tokens=max_new, do_sample=False,
        pad_token_id=tok.eos_token_id,
    )
    ans = tok.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()
    for stop in ['<|im_end|>', '<|im_start|>', '<|endoftext|>']:
        if stop in ans: ans = ans[:ans.index(stop)].strip()
    return ans
# Speed test
log("Speed test...")
test_cands = val_cands_all.get(0, get_same_lang_candidates(val_qs[0], val_emb[0],
              str(val_df.iloc[0]['subset'])))
t0 = time.time()
test_ans = ft_generate(val_qs[0], 'English', test_cands)
log(f"Speed: {time.time()-t0:.1f}s per query. Answer: {test_ans[:100]}...")
# ===========================================================================
# STEP 2: VAL GENERATION (400 samples, ~15-30 min)
# ===========================================================================
log(f"\n{'='*60}")
log("STEP 2: Val generation (400 samples)")
log(f"{'='*60}")
rng = np.random.default_rng(42)
val_sample = []
for sub in SUBSET_TO_LANG:
    idxs = [i for i in range(len(val_df))
            if str(val_df.iloc[i]['subset']) == sub and i in val_cands_all and val_cands_all[i]]
    val_sample += [int(x) for x in rng.choice(idxs, min(50, len(idxs)), replace=False)]
log(f"Val sample: {len(val_sample)} queries")
fprog = OUTPUT_DIR / 'ftqwen_val_sample.json'
fans = json.load(open(fprog)) if fprog.exists() else {}
log(f"Already generated: {len(fans)}")
t0 = time.time()
for n, i in enumerate(tqdm(val_sample, desc="FT-Qwen val")):
    if str(i) in fans:
        continue
    sub = str(val_df.iloc[i]['subset'])
    cands = val_cands_all.get(i, get_same_lang_candidates(val_qs[i], val_emb[i], sub))
    if not cands:
        fans[str(i)] = ""
        continue
    fans[str(i)] = ft_generate(val_qs[i], SUBSET_TO_LANG[sub], cands)
    if (n + 1) % 25 == 0:
        json.dump(fans, open(fprog, 'w'))
        elapsed = time.time() - t0
        remaining = elapsed / max(n + 1, 1) * (len(val_sample) - n - 1)
        log(f"  {n+1}/{len(val_sample)} done, ~{remaining/60:.0f}min remaining")
json.dump(fans, open(fprog, 'w'))
log(f"Val generation done: {len(fans)} answers in {(time.time()-t0)/60:.1f}min")
# ===========================================================================
# STEP 3: EVALUATE — Generated vs Retrieval per language (ROUGE)
# ===========================================================================
log(f"\n{'='*60}")
log("STEP 3: Per-language evaluation (ROUGE)")
log(f"{'='*60}")
TEST_MIX = {
    'Eng_Uga': 0.284, 'Aka_Gha': 0.188, 'Eng_Gha': 0.188,
    'Lug_Uga': 0.143, 'Swa_Ken': 0.087, 'Eng_Ken': 0.064,
    'Amh_Eth': 0.023, 'Eng_Eth': 0.023,
}
per_lang = defaultdict(lambda: {'ret_r1': [], 'ret_rl': [], 'gen_r1': [], 'gen_rl': []})
for i in val_sample:
    ref = str(val_df.iloc[i]['output']).strip()
    sub = str(val_df.iloc[i]['subset'])
    if not ref:
        continue
    rt = uni_toks(ref)
    # Retrieval top-1
    cands = val_cands_all.get(i)
    if cands:
        ret = cands[0]['answer']
        per_lang[sub]['ret_r1'].append(uni_r1(rt, uni_toks(ret)))
        per_lang[sub]['ret_rl'].append(uni_rl(rt, uni_toks(ret)))
    # Generated
    gen = fans.get(str(i), '')
    if gen:
        per_lang[sub]['gen_r1'].append(uni_r1(rt, uni_toks(gen)))
        per_lang[sub]['gen_rl'].append(uni_rl(rt, uni_toks(gen)))
log(f"\n{'Sub':<12} {'Ret R1':>8} {'Gen R1':>8} {'Δ R1':>7} "
    f"{'Ret RL':>8} {'Gen RL':>8} {'Δ RL':>7} {'Use Gen?':>9}")
log('-' * 80)
gen_better_r1 = {}
gen_better_rl = {}
for sub in sorted(SUBSET_TO_LANG.keys()):
    rr1 = np.mean(per_lang[sub]['ret_r1']) if per_lang[sub]['ret_r1'] else 0
    rrl = np.mean(per_lang[sub]['ret_rl']) if per_lang[sub]['ret_rl'] else 0
    gr1 = np.mean(per_lang[sub]['gen_r1']) if per_lang[sub]['gen_r1'] else 0
    grl = np.mean(per_lang[sub]['gen_rl']) if per_lang[sub]['gen_rl'] else 0
    gen_better_r1[sub] = gr1 > rr1 + 0.005
    gen_better_rl[sub] = grl > rrl + 0.005
    verdict = "GEN" if gen_better_r1[sub] or gen_better_rl[sub] else "RET"
    log(f"  {sub:<12} {rr1:>8.4f} {gr1:>8.4f} {gr1-rr1:>+7.4f} "
        f"{rrl:>8.4f} {grl:>8.4f} {grl-rrl:>+7.4f} {verdict:>9}")
# ===========================================================================
# STEP 4: GENERATE TEST ANSWERS (only needed languages)
# ===========================================================================
log(f"\n{'='*60}")
log("STEP 4: Test generation (2618 queries)")
log(f"{'='*60}")
test_gen_path = OUTPUT_DIR / 'ftqwen_test.json'
test_gen = json.load(open(test_gen_path)) if test_gen_path.exists() else {}
log(f"Already generated: {len(test_gen)}")
t0 = time.time()
for i in tqdm(range(len(test_df)), desc="FT-Qwen test"):
    rid = str(test_df.iloc[i]['ID'])
    if rid in test_gen:
        continue
    q = test_qs[i].strip()
    sub = test_subs[i]
    lang = SUBSET_TO_LANG.get(sub, sub)
    cands = get_same_lang_candidates(q, test_emb[i], sub, k=K_CANDIDATES, exclude_exact=False)
    if not cands:
        test_gen[rid] = "No answer."
        continue
    test_gen[rid] = ft_generate(q, lang, cands)
    if (i + 1) % 100 == 0:
        json.dump(test_gen, open(test_gen_path, 'w'))
        elapsed = time.time() - t0
        done = sum(1 for _ in test_gen if _ not in {})
        log(f"  {i+1}/{len(test_df)}, ~{elapsed/60:.0f}min elapsed")
json.dump(test_gen, open(test_gen_path, 'w'))
log(f"Test generation done: {len(test_gen)} answers in {(time.time()-t0)/60:.1f}min")
# Free GPU
del ft, base
gc.collect(); torch.cuda.empty_cache()
log("GPU freed.")
# ===========================================================================
# STEP 5: BUILD SUBMISSIONS
# ===========================================================================
log(f"\n{'='*60}")
log("STEP 5: Build submissions")
log(f"{'='*60}")
rows_v1 = []  # Safe: MBR for ROUGE, FT-Qwen for LLM
rows_v2 = []  # Compliant: FT-Qwen everywhere
rows_v3 = []  # Compliant: MBR everywhere (fallback)
for i in tqdm(range(len(test_df)), desc="Building submissions"):
    q = test_qs[i].strip()
    sub = test_subs[i]
    rid = str(test_df.iloc[i]['ID'])
    pool_type, alpha, margin = choice.get(sub, ('2leg', 0.05, 99.0))
    # Retrieval candidates
    if pool_type == '4leg':
        cands = union4(q, test_emb[i], gem_test[i], bge_test[i], sub, exclude_exact=False)
    else:
        cands = get_same_lang_candidates(q, test_emb[i], sub, k=K_CANDIDATES, exclude_exact=False)
    if not cands:
        for rows in [rows_v1, rows_v2, rows_v3]:
            rows.append({'ID': test_df.iloc[i]['ID'],
                         'TargetR1F1': 'No answer', 'TargetRLF1': 'No answer',
                         'TargetLLM': 'No answer'})
        continue
    # MBR answers (proven for ROUGE)
    dd, w, u1, uL = uni_prep(cands)
    ans_mbr_r1 = dd[mbr_idx(u1, w, alpha, margin)]
    ans_mbr_rl = dd[mbr_idx(uL, w, alpha, margin)]
    # Stitch for R1 where gated
    sg = uni_stitch_gate.get(sub, {})
    if sg.get('use', False):
        stitch_cands = cands
        if sg.get('pool') == '4leg' and pool_type != '4leg':
            stitch_cands = union4(q, test_emb[i], gem_test[i], bge_test[i], sub, exclude_exact=False)
        ans_stitch_r1 = uni_stitch(stitch_cands, sg['lam'], sub)
    else:
        ans_stitch_r1 = ans_mbr_r1
    # FT-Qwen answer
    gen = test_gen.get(rid, dd[0])
    # V1: SAFE SPLIT — proven ROUGE + FT-Qwen LLM
    rows_v1.append({
        'ID': test_df.iloc[i]['ID'],
        'TargetR1F1': ans_stitch_r1,
        'TargetRLF1': ans_mbr_rl,
        'TargetLLM': gen,
    })
    # V2: COMPLIANT — FT-Qwen for all
    rows_v2.append({
        'ID': test_df.iloc[i]['ID'],
        'TargetR1F1': gen, 'TargetRLF1': gen, 'TargetLLM': gen,
    })
    # V3: COMPLIANT — MBR for all (current baseline)
    rows_v3.append({
        'ID': test_df.iloc[i]['ID'],
        'TargetR1F1': ans_mbr_r1, 'TargetRLF1': ans_mbr_r1, 'TargetLLM': ans_mbr_r1,
    })
# Save
for fname, rows in [
    ('submission_ft_safe_split.csv', rows_v1),
    ('submission_ft_compliant_gen.csv', rows_v2),
    ('submission_ft_compliant_mbr.csv', rows_v3),
]:
    df = pd.DataFrame(rows)[SUB_COLS]
    assert len(df) == len(sample_sub), f"{fname}: {len(df)} vs {len(sample_sub)}"
    df.to_csv(OUTPUT_DIR / fname, index=False)
    log(f"Saved: {fname}")
# ===========================================================================
# SUMMARY
# ===========================================================================
log(f"\n{'='*60}")
log("DONE — SUBMISSION RECOMMENDATIONS")
log(f"{'='*60}")
log(f"""
1. submission_ft_safe_split.csv    ← SUBMIT FIRST
   R1=stitch/MBR (retrieval), RL=MBR (retrieval), LLM=FT-Qwen (generated)
   Why: keeps proven ROUGE, upgrades LLM column
2. submission_ft_compliant_gen.csv ← SUBMIT IF #1 improves over 0.6670
   All columns = FT-Qwen generated (fully open-source + compliant)
3. submission_ft_compliant_mbr.csv ← SAFETY NET
   All columns = MBR retrieval (proven baseline, compliant)
Current LB: 0.6670 | Leader: 0.7285
""")


[05:52:04] Loading fine-tuned Qwen in 4-bit (fast inference)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

[05:53:11] Model loaded in 4-bit. GPU mem: 5.7GB
[05:53:11] Speed test...


AttributeError: 'list' object has no attribute 'get'

In [ ]:
# Fix: val_cands_all is a list, not dict
def get_val_cands(i):
    try:
        c = val_cands_all[i]
        if c: return c
    except: pass
    return get_same_lang_candidates(val_qs[i], val_emb[i], str(val_df.iloc[i]['subset']))

# Speed test
log("Speed test...")
test_cands = get_val_cands(0)
t0 = time.time()
test_ans = ft_generate(val_qs[0], 'English', test_cands)
log(f"Speed: {time.time()-t0:.1f}s per query")
log(f"Answer preview: {test_ans[:150]}...")

# Val generation
rng = np.random.default_rng(42)
val_sample = []
for sub in SUBSET_TO_LANG:
    idxs = [i for i in range(len(val_df)) if str(val_df.iloc[i]['subset'])==sub and val_cands_all[i]]
    val_sample += [int(x) for x in rng.choice(idxs, min(50, len(idxs)), replace=False)]
log(f"Val sample: {len(val_sample)}")

fprog = OUTPUT_DIR / 'ftqwen_val_sample.json'
fans = json.load(open(fprog)) if fprog.exists() else {}
log(f"Already cached: {len(fans)}")

t0 = time.time()
for n, i in enumerate(tqdm(val_sample, desc="FT-Qwen val")):
    if str(i) in fans: continue
    fans[str(i)] = ft_generate(val_qs[i], SUBSET_TO_LANG[str(val_df.iloc[i]['subset'])], get_val_cands(i))
    if (n+1) % 25 == 0:
        json.dump(fans, open(fprog, 'w'))
json.dump(fans, open(fprog, 'w'))
log(f"Val done: {len(fans)} in {(time.time()-t0)/60:.1f}min")

[15:27:58] Speed test...


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


[15:28:12] Speed: 14.4s per query
[15:28:12] Answer preview: Nnwomasua ne adwuma nteteeɛ boa mmabun a wɔ hia ne aɛma sɛ wɔbɔ wɔn ho ban afi wɔn a wɔte wɔn so, atubrafo, anaa wɔn a wɔte wɔn so wɔ mpɔtam hɔ, atubr...
[15:28:13] Val sample: 400
[15:28:13] Already cached: 0



FT-Qwen val: 100%|██████████| 400/400 [1:14:58<00:00, 11.25s/it]

[16:43:12] Val done: 400 in 75.0min


In [ ]:
# === EVALUATE: Generated vs Retrieval ===
per_lang = defaultdict(lambda: {'ret_r1':[], 'ret_rl':[], 'gen_r1':[], 'gen_rl':[]})

for i in val_sample:
    ref = str(val_df.iloc[i]['output']).strip()
    sub = str(val_df.iloc[i]['subset'])
    if not ref: continue
    rt = uni_toks(ref)

    cands = get_val_cands(i)
    if cands:
        ret = cands[0]['answer']
        per_lang[sub]['ret_r1'].append(uni_r1(rt, uni_toks(ret)))
        per_lang[sub]['ret_rl'].append(uni_rl(rt, uni_toks(ret)))

    gen = fans.get(str(i), '')
    if gen:
        per_lang[sub]['gen_r1'].append(uni_r1(rt, uni_toks(gen)))
        per_lang[sub]['gen_rl'].append(uni_rl(rt, uni_toks(gen)))

TEST_MIX = {'Eng_Uga':0.284,'Aka_Gha':0.188,'Eng_Gha':0.188,'Lug_Uga':0.143,
            'Swa_Ken':0.087,'Eng_Ken':0.064,'Amh_Eth':0.023,'Eng_Eth':0.023}

print(f"\n{'Sub':<12} {'Ret R1':>8} {'Gen R1':>8} {'Δ R1':>7}  {'Ret RL':>8} {'Gen RL':>8} {'Δ RL':>7}")
print('-'*75)
tw_ret_r1, tw_gen_r1, tw_ret_rl, tw_gen_rl = 0,0,0,0
for sub in sorted(SUBSET_TO_LANG.keys()):
    rr1 = np.mean(per_lang[sub]['ret_r1']) if per_lang[sub]['ret_r1'] else 0
    rrl = np.mean(per_lang[sub]['ret_rl']) if per_lang[sub]['ret_rl'] else 0
    gr1 = np.mean(per_lang[sub]['gen_r1']) if per_lang[sub]['gen_r1'] else 0
    grl = np.mean(per_lang[sub]['gen_rl']) if per_lang[sub]['gen_rl'] else 0
    w = TEST_MIX.get(sub, 0)
    tw_ret_r1 += w*rr1; tw_gen_r1 += w*gr1; tw_ret_rl += w*rrl; tw_gen_rl += w*grl
    m = " ★" if gr1 > rr1+0.005 else ""
    print(f"  {sub:<12} {rr1:>8.4f} {gr1:>8.4f} {gr1-rr1:>+7.4f}  {rrl:>8.4f} {grl:>8.4f} {grl-rrl:>+7.4f}{m}")

print(f"\n  Test-weighted:")
print(f"  Retrieval: R1={tw_ret_r1:.4f}  RL={tw_ret_rl:.4f}")
print(f"  Generated: R1={tw_gen_r1:.4f}  RL={tw_gen_rl:.4f}")
print(f"  Delta:     R1={tw_gen_r1-tw_ret_r1:+.4f}  RL={tw_gen_rl-tw_ret_rl:+.4f}")

# Estimated scores (0.37*R1 + 0.37*RL + 0.26*LLM)
curr_llm = 0.785  # current LLM column on LB
ft_llm_est = 0.82  # conservative estimate for fine-tuned
print(f"\n  Current LB (split):  {0.37*tw_ret_r1 + 0.37*tw_ret_rl + 0.26*curr_llm:.4f}")
print(f"  V1 (ret ROUGE + ft LLM): {0.37*tw_ret_r1 + 0.37*tw_ret_rl + 0.26*ft_llm_est:.4f}")
print(f"  V2 (all generated):      {0.37*tw_gen_r1 + 0.37*tw_gen_rl + 0.26*ft_llm_est:.4f}")

print(f"\n  Recommendation based on ROUGE comparison:")
if tw_gen_r1 > tw_ret_r1 + 0.01:
    print("  → Generation BEATS retrieval on ROUGE! Submit V2 (all-generated)")
else:
    print("  → Retrieval wins ROUGE (expected). Submit V1 (ret ROUGE + ft LLM)")

NameError: name 'defaultdict' is not defined

In [ ]:
# ---- A) ENG_GHA FULL-VAL: generation vs current-best, split-half ----
eg_idxs = [i for i in range(len(val_df)) if str(val_df.iloc[i]['subset'])=='Eng_Gha'
           and val_cands_all[i] and str(val_df.iloc[i]['output']).strip()]
egprog = OUTPUT_DIR / 'ftqwen_enggha_val.json'
eg = json.load(open(egprog)) if egprog.exists() else {}
for n, i in enumerate(tqdm(eg_idxs, desc="Gen Eng_Gha val")):
    if str(i) in eg: continue
    eg[str(i)] = ft_generate(val_qs[i], 'English (Ghana)', val_cands_all[i])
    if (n+1) % 50 == 0: json.dump(eg, open(egprog, 'w'))
json.dump(eg, open(egprog, 'w'))

hold = eg_idxs[1::2]   # untouched half
def cur_best_r1(i):    # current pipeline: stitch on 4leg
    return uni_r1(uni_toks(str(val_df.iloc[i]['output']))[:CAP],
                  uni_toks(uni_stitch(v4c[i] if v4c[i] else val_cands_all[i], 0.85, 'Eng_Gha'))[:CAP])
g_r1 = np.mean([uni_r1(uni_toks(str(val_df.iloc[i]['output']))[:CAP], uni_toks(eg[str(i)])[:CAP]) for i in hold])
g_rl = np.mean([uni_rl(uni_toks(str(val_df.iloc[i]['output']))[:CAP], uni_toks(eg[str(i)])[:CAP]) for i in hold])
c_r1 = np.mean([cur_best_r1(i) for i in hold])
print(f"Eng_Gha HOLDOUT (n={len(hold)}):  gen R1={g_r1:.4f} RL={g_rl:.4f}  |  current-best R1={c_r1:.4f}")
print("ADOPT gen for Eng_Gha ROUGE columns if gen R1 > current + 0.01 (transfer discount applied).")


Gen Eng_Gha val: 100%|██████████| 1104/1104 [2:41:13<00:00,  8.76s/it]


Eng_Gha HOLDOUT (n=552):  gen R1=0.4230 RL=0.3109  |  current-best R1=0.3676
ADOPT gen for Eng_Gha ROUGE columns if gen R1 > current + 0.01 (transfer discount applied).


In [ ]:
# pre-flight: every name the mega-cell touches
for nm in ['ft','tok','val_df','val_qs','val_cands_all','val_refscores','P','choice',
           'uni_stitch_gate','test_df','test_qs','test_subs','test_emb','gem_test',
           'bge_test','SUB_COLS','OUTPUT_DIR','union4','get_same_lang_candidates',
           'uni_prep','mbr_idx','uni_stitch','uni_r1','uni_rl','uni_toks','CAP','K_CANDIDATES']:
    assert nm in dir() or nm in globals(), f"MISSING: {nm}"
print("All names present — clear to launch.")

All names present — clear to launch.


In [ ]:
# =============================================================================
# OVERNIGHT MEGA-CELL: rubric gate -> Aka_Gha probe -> FT2 leg -> test gen -> V1/V2
# Fully resumable. Requires: bootstrap state + ft/tok loaded (PeftModel).
# =============================================================================
import json, re as _re2, torch, pickle
import numpy as np, pandas as pd
from tqdm import tqdm

# ---------- shared helpers ----------
@torch.no_grad()
def rubric_judge(q, ref, cand):
    msgs = [{"role":"system","content":
             "You grade answers to health questions. Score the candidate 1-5 considering: "
             "factual accuracy vs the reference, completeness, and language appropriateness "
             "(same language as the question). Reply with ONLY the integer."},
            {"role":"user","content": f"Question: {q}\n\nReference answer:\n{ref}\n\nCandidate answer:\n{cand}\n\nScore:"}]
    enc = tok.apply_chat_template(msgs, add_generation_prompt=True,
                                  return_tensors='pt', return_dict=True).to('cuda:0')
    out = ft.generate(**enc, max_new_tokens=4, do_sample=False, pad_token_id=tok.eos_token_id)
    m = _re2.search(r'[1-5]', tok.decode(out[0][enc['input_ids'].shape[1]:], skip_special_tokens=True))
    return int(m.group()) if m else None

@torch.no_grad()
def ft_generate(q, lang, cands, max_new=350):
    ctx = "\n".join(f"{k+1}. {c['answer']}" for k, c in enumerate(cands[:5]))
    msgs = [{"role":"system","content":
             f"You are a multilingual health expert. Answer health questions based on the "
             f"reference information provided. Use the EXACT words and phrases from the "
             f"references when possible. Be complete and accurate. Answer in {lang}."},
            {"role":"user","content": f"Question: {q}\n\nReference answers:\n{ctx}"}]
    enc = tok.apply_chat_template(msgs, add_generation_prompt=True,
                                  return_tensors='pt', return_dict=True).to('cuda:0')
    out = ft.generate(**enc, max_new_tokens=max_new, do_sample=False, pad_token_id=tok.eos_token_id)
    return tok.decode(out[0][enc['input_ids'].shape[1]:], skip_special_tokens=True).strip()

def save_json(obj, p): json.dump(obj, open(p, 'w'))

# =============================================================================
# STAGE 1: RUBRIC JUDGING -> FT_LANGS (cached: rubric_scores.json)
# =============================================================================
print("="*60 + "\nSTAGE 1: Rubric judging\n" + "="*60)
fans = json.load(open(OUTPUT_DIR / 'ftqwen_val_sample.json'))
gpath = OUTPUT_DIR / 'gemini_val_sample.json'
gem_vans = json.load(open(gpath)) if gpath.exists() else {}
rng = np.random.default_rng(42)
val_sample = []
for sub in SUBSET_TO_LANG:
    idxs = [i for i in range(len(val_df)) if str(val_df.iloc[i]['subset'])==sub and val_cands_all[i]]
    val_sample += [int(x) for x in rng.choice(idxs, min(50, len(idxs)), replace=False)]

jpath = OUTPUT_DIR / 'rubric_scores.json'
jscores = json.load(open(jpath)) if jpath.exists() else {}
todo = [i for i in val_sample if str(i) in fans and str(i) not in jscores
        and str(val_df.iloc[i]['output']).strip()]
for n, i in enumerate(tqdm(todo, desc="S1 rubric")):
    q = val_qs[i]; ref = str(val_df.iloc[i]['output']).strip()
    s = {'top1': rubric_judge(q, ref, val_cands_all[i][0]['answer']),
         'ft':   rubric_judge(q, ref, fans[str(i)])}
    g = gem_vans.get(str(i), '')
    if g: s['gem'] = rubric_judge(q, ref, g)
    jscores[str(i)] = s
    if (n+1) % 25 == 0: save_json(jscores, jpath)
save_json(jscores, jpath)

MARGIN = 0.4
FT_LANGS = set()
print(f"{'Sub':<10} {'n':>4} {'top1':>6} {'gem':>6} {'ft':>6} {'ft-top1':>8}  LLM")
for sub in sorted(SUBSET_TO_LANG):
    rows = [jscores[str(i)] for i in val_sample
            if str(val_df.iloc[i]['subset'])==sub and str(i) in jscores
            and jscores[str(i)].get('top1') is not None and jscores[str(i)].get('ft') is not None]
    if not rows: continue
    t1 = np.mean([r['top1'] for r in rows]); ftm = np.mean([r['ft'] for r in rows])
    gems = [r['gem'] for r in rows if r.get('gem') is not None]
    gm = np.mean(gems) if gems else float('nan')
    if ftm - t1 >= MARGIN: FT_LANGS.add(sub)
    print(f"{sub:<10} {len(rows):>4} {t1:>6.2f} {gm:>6.2f} {ftm:>6.2f} {ftm-t1:>+8.2f}  "
          f"{'FT' if sub in FT_LANGS else 'ret'}")
save_json(sorted(FT_LANGS), OUTPUT_DIR / 'ft_langs.json')
print(f"FT_LANGS = {sorted(FT_LANGS)}")

# =============================================================================
# STAGE 2: AKA_GHA FULL-VAL GEN PROBE (cached: ftqwen_akagha_val.json)
# =============================================================================
print("\n" + "="*60 + "\nSTAGE 2: Aka_Gha generation probe\n" + "="*60)
ak_idxs = [i for i in range(len(val_df)) if str(val_df.iloc[i]['subset'])=='Aka_Gha'
           and val_cands_all[i] and str(val_df.iloc[i]['output']).strip()]
akp = OUTPUT_DIR / 'ftqwen_akagha_val.json'
ak = json.load(open(akp)) if akp.exists() else {}
for n, i in enumerate(tqdm(ak_idxs, desc="S2 Aka_Gha gen")):
    if str(i) in ak: continue
    ak[str(i)] = ft_generate(val_qs[i], 'Akan (Ghana)', val_cands_all[i])
    if (n+1) % 50 == 0: save_json(ak, akp)
save_json(ak, akp)
hold = ak_idxs[1::2]
g_r1 = np.mean([uni_r1(uni_toks(str(val_df.iloc[i]['output']))[:CAP], uni_toks(ak[str(i)])[:CAP]) for i in hold])
g_rl = np.mean([uni_rl(uni_toks(str(val_df.iloc[i]['output']))[:CAP], uni_toks(ak[str(i)])[:CAP]) for i in hold])
c_r1 = np.mean([uni_r1(uni_toks(str(val_df.iloc[i]['output']))[:CAP],
        uni_toks(uni_stitch(val_cands_all[i], 0.70, 'Aka_Gha'))[:CAP]) for i in hold])
c_rl = np.mean([val_refscores and 0 or 0])  # placeholder, real RL below
# current-best RL for Aka_Gha = MBR-rougeL on 2leg (from P cache)
pr, rf = P['2leg']['prep'], P['2leg']['ref']
c_rl = np.mean([rf[i][mbr_idx(pr[i][3], pr[i][1], 0.10, 0.010)][1] for i in hold if rf[i]])
AKA_GEN_R1 = g_r1 > c_r1 + 0.01
AKA_GEN_RL = g_rl > c_rl + 0.01
print(f"Aka_Gha holdout (n={len(hold)}): gen R1={g_r1:.4f} vs cur {c_r1:.4f} -> {'ADOPT' if AKA_GEN_R1 else 'keep'}")
print(f"                                 gen RL={g_rl:.4f} vs cur {c_rl:.4f} -> {'ADOPT' if AKA_GEN_RL else 'keep'}")
save_json({'r1': bool(AKA_GEN_R1), 'rl': bool(AKA_GEN_RL)}, OUTPUT_DIR / 'aka_gen_gate.json')

# =============================================================================
# STAGE 3: FT2-GHA LEG CHECK (CPU; quick; informational for ROUGE pools)
# =============================================================================
print("\n" + "="*60 + "\nSTAGE 3: FT2-Gha leg (skipped if gen adopted for both Gha)\n" + "="*60)
# If generation takes over Eng_Gha and Aka_Gha ROUGE columns, FT2's leg is moot. Check cheaply:
if AKA_GEN_R1 and AKA_GEN_RL:
    print("Generation adopted for Aka_Gha both columns; Eng_Gha already adopted. FT2 leg retired.")
else:
    print("FT2 leg still relevant for non-adopted columns — flagging for tomorrow (CPU cell exists).")

# =============================================================================
# STAGE 4: FT TEST GENERATION for all adopted uses (cached: ftqwen_test_answers.json)
# Languages needed: Eng_Gha (ROUGE confirmed), Aka_Gha (if gate), FT_LANGS (LLM column)
# =============================================================================
print("\n" + "="*60 + "\nSTAGE 4: FT test-set generation\n" + "="*60)
GEN_ROUGE_LANGS = {'Eng_Gha'} | ({'Aka_Gha'} if (AKA_GEN_R1 or AKA_GEN_RL) else set())
NEED = GEN_ROUGE_LANGS | FT_LANGS
print(f"Generating test answers for: {sorted(NEED)}")
ftest = OUTPUT_DIR / 'ftqwen_test_answers.json'
ft_test_ans = json.load(open(ftest)) if ftest.exists() else {}
todo = [i for i in range(len(test_df)) if test_subs[i] in NEED
        and str(test_df.iloc[i]['ID']) not in ft_test_ans]
for n, i in enumerate(tqdm(todo, desc="S4 test gen")):
    rid = str(test_df.iloc[i]['ID'])
    cands = get_same_lang_candidates(test_qs[i].strip(), test_emb[i], test_subs[i],
                                     exclude_exact=False)
    ft_test_ans[rid] = ft_generate(test_qs[i], SUBSET_TO_LANG[test_subs[i]], cands) if cands else "No answer."
    if (n+1) % 50 == 0: save_json(ft_test_ans, ftest)
save_json(ft_test_ans, ftest)
print(f"Test answers: {len(ft_test_ans)}")



STAGE 1: Rubric judging


S1 rubric: 0it [00:00, ?it/s]


Sub           n   top1    gem     ft  ft-top1  LLM
Aka_Gha      50   4.18   4.14   4.18    +0.00  ret
Amh_Eth      50   4.48   4.44   4.30    -0.18  ret
Eng_Eth      48   4.38   4.33   4.46    +0.08  ret
Eng_Gha      50   4.04   3.72   4.08    +0.04  ret
Eng_Ken      47   4.00   3.98   3.94    -0.06  ret
Eng_Uga      40   4.08   3.90   4.03    -0.05  ret
Lug_Uga      50   4.64   4.16   4.26    -0.38  ret
Swa_Ken      50   4.82   4.30   4.54    -0.28  ret
FT_LANGS = []

STAGE 2: Aka_Gha generation probe


S2 Aka_Gha gen:   0%|          | 0/1114 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
S2 Aka_Gha gen: 100%|██████████| 1114/1114 [2:34:44<00:00,  8.33s/it]


Aka_Gha holdout (n=557): gen R1=0.3173 vs cur 0.3839 -> keep
                                 gen RL=0.2256 vs cur 0.2068 -> ADOPT

STAGE 3: FT2-Gha leg (skipped if gen adopted for both Gha)
FT2 leg still relevant for non-adopted columns — flagging for tomorrow (CPU cell exists).

STAGE 4: FT test-set generation
Generating test answers for: ['Aka_Gha', 'Eng_Gha']


S4 test gen:   8%|▊         | 83/983 [47:51<8:38:52, 34.59s/it]


KeyboardInterrupt: 

In [23]:
!pip install -q -U torchao pylcs

In [ ]:
FT_MODEL_DIR = OUTPUT_DIR / 'qwen-ft-health'

In [ ]:
import torch, gc
try: del ft, base
except NameError: pass
gc.collect(); torch.cuda.empty_cache()

from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
tok = AutoTokenizer.from_pretrained(str(FT_MODEL_DIR))
base = AutoModelForCausalLM.from_pretrained('Qwen/Qwen2.5-7B-Instruct',
        dtype=torch.bfloat16, device_map='cuda:0')
ft = PeftModel.from_pretrained(base, str(FT_MODEL_DIR)); ft.eval()
print(f"bf16 loaded — GPU free: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

bf16 loaded — GPU free: 7.9 GB


In [ ]:
# ==== STAGE 4 (FAST): batched FT test generation, resumes from cache ====
import json, torch
from tqdm import tqdm
tok.padding_side = 'left'   # required for decoder-only batched generation
if tok.pad_token is None: tok.pad_token = tok.eos_token

NEED = {'Eng_Gha', 'Aka_Gha'}
ftest = OUTPUT_DIR / 'ftqwen_test_answers.json'
ft_test_ans = json.load(open(ftest)) if ftest.exists() else {}
todo = [i for i in range(len(test_df)) if test_subs[i] in NEED
        and str(test_df.iloc[i]['ID']) not in ft_test_ans]
print(f"Remaining: {len(todo)}")

def prompt_text(i, cands):
    ctx = "\n".join(f"{k+1}. {c['answer']}" for k, c in enumerate(cands[:5]))
    msgs = [{"role":"system","content":
             f"You are a multilingual health expert. Answer health questions based on the "
             f"reference information provided. Use the EXACT words and phrases from the "
             f"references when possible. Be complete and accurate. Answer in {SUBSET_TO_LANG[test_subs[i]]}."},
            {"role":"user","content": f"Question: {test_qs[i]}\n\nReference answers:\n{ctx}"}]
    return tok.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False)

pending.sort(key=lambda x: len(x[1]))
B = 6
pending = []
for i in todo:
    cands = get_same_lang_candidates(test_qs[i].strip(), test_emb[i], test_subs[i],
                                     exclude_exact=False)
    if not cands:
        ft_test_ans[str(test_df.iloc[i]['ID'])] = "No answer."; continue
    pending.append((i, prompt_text(i, cands)))

with torch.no_grad():
    for s in tqdm(range(0, len(pending), B), desc="S4 batched"):
        chunk = pending[s:s+B]
        enc = tok([p for _, p in chunk], return_tensors='pt',
                  padding=True, truncation=True, max_length=1400).to('cuda:0')
        out = ft.generate(**enc, max_new_tokens=350, do_sample=False,
                          pad_token_id=tok.pad_token_id)
        for (i, _), o in zip(chunk, out):
            ans = tok.decode(o[enc['input_ids'].shape[1]:], skip_special_tokens=True).strip()
            ft_test_ans[str(test_df.iloc[i]['ID'])] = ans
        if (s // B) % 5 == 0: json.dump(ft_test_ans, open(ftest, 'w'))
json.dump(ft_test_ans, open(ftest, 'w'))
print(f"Done: {len(ft_test_ans)} test answers")
tok.padding_side = 'right'

Remaining: 873


S4 batched: 100%|██████████| 146/146 [1:04:32<00:00, 26.53s/it]

Done: 983 test answers


In [ ]:
# ==== Gates: the validated decisions from last night's run ====
AKA_GEN_R1 = False   # Aka_Gha R1 stays stitch (gen lost: 0.317 vs 0.384)
AKA_GEN_RL = True    # Aka_Gha RL switches to generation (gen won: 0.226 vs 0.207)
FT_LANGS   = set()   # LLM column: no language cleared the rubric margin -> retrieval everywhere

In [ ]:
# =============================================================================
# STAGE 5: BUILD V1 (split, all adopted pieces) + V2 (compliant identical)
# =============================================================================
print("\n" + "="*60 + "\nSTAGE 5: Build submissions\n" + "="*60)
rows_v1, rows_v2 = [], []
for i in tqdm(range(len(test_df)), desc="S5 assemble"):
    rid = str(test_df.iloc[i]['ID']); sub = test_subs[i]
    tag, a, m = choice.get(sub, ('2leg', 0.05, 99.0))
    if tag == '4leg':
        pool = union4(test_qs[i].strip(), test_emb[i], gem_test[i], bge_test[i], sub, exclude_exact=False)
    else:
        pool = get_same_lang_candidates(test_qs[i].strip(), test_emb[i], sub,
                                        k=K_CANDIDATES, exclude_exact=False)
    if not pool:
        for R in (rows_v1, rows_v2):
            R.append({'ID': test_df.iloc[i]['ID'], 'TargetR1F1': 'No answer',
                      'TargetRLF1': 'No answer', 'TargetLLM': 'No answer'})
        continue
    dd, ddw, u1, uL = uni_prep(pool)
    r1a = dd[mbr_idx(u1, ddw, a, m)]; rla = dd[mbr_idx(uL, ddw, a, m)]
    g = uni_stitch_gate.get(sub)
    if g and g['use']: r1a = uni_stitch(pool, g['lam'], sub)
    gen = ft_test_ans.get(rid, '')
    if sub == 'Eng_Gha' and gen: r1a, rla = gen, gen          # confirmed both columns
    if sub == 'Aka_Gha' and gen:
        if AKA_GEN_R1: r1a = gen
        if AKA_GEN_RL: rla = gen
    llm = gen if (sub in FT_LANGS and gen) else pool[0]['answer']
    rows_v1.append({'ID': test_df.iloc[i]['ID'], 'TargetR1F1': r1a, 'TargetRLF1': rla, 'TargetLLM': llm})
    # V2 compliant: gen where gen owns both ROUGE columns (it's one answer + open-source); else combined MBR
    if sub == 'Eng_Gha' and gen: comb = gen
    elif sub == 'Aka_Gha' and gen and AKA_GEN_R1 and AKA_GEN_RL: comb = gen
    else: comb = dd[mbr_idx(0.5*u1 + 0.5*uL, ddw, a, m)]
    rows_v2.append({'ID': test_df.iloc[i]['ID'], 'TargetR1F1': comb, 'TargetRLF1': comb, 'TargetLLM': comb})

for name, rows in (('submission_v1_gen.csv', rows_v1), ('submission_v2_compliant.csv', rows_v2)):
    df = pd.DataFrame(rows).reindex(columns=SUB_COLS)
    assert len(df) == len(test_df) and not df.isnull().any().any()
    df.to_csv(OUTPUT_DIR / name, index=False)
    print(f"Saved: {name}")

print("\nDONE. Morning: check Stage 1 table + Stage 2 gates above, then submit V1, then V2.")


STAGE 5: Build submissions


S5 assemble: 100%|██████████| 2618/2618 [00:53<00:00, 49.15it/s] 


Saved: submission_v1_gen.csv
Saved: submission_v2_compliant.csv

DONE. Morning: check Stage 1 table + Stage 2 gates above, then submit V1, then V2.


In [ ]:
# =============================================================================
# V3: BEST SINGLE ANSWER PER LANGUAGE (identical columns, open-source)
# Strategies: comb_mbr (V2's winner), stitch, ft-gen (where cached)
# Scored on val holdout at 0.37*R1 + 0.37*RL; judge assumed ~equal (V2 evidence:
# consensus answers judge WELL, 0.8052) with a small penalty for stitch.
# =============================================================================
import json
import numpy as np, pandas as pd
from tqdm import tqdm

eg = json.load(open(OUTPUT_DIR / 'ftqwen_enggha_val.json'))
ak = json.load(open(OUTPUT_DIR / 'ftqwen_akagha_val.json'))
gen_val = {'Eng_Gha': eg, 'Aka_Gha': ak}

STITCH_PENALTY = 0.005   # frankenstein answers may judge slightly worse; small prior

def ref_t(i): return uni_toks(str(val_df.iloc[i]['output']))[:CAP]
def score_pair(rt, ans):
    at = uni_toks(ans)[:CAP]
    return 0.37*uni_r1(rt, at) + 0.37*uni_rl(rt, at)

print(f"{'Sub':<10} {'comb_mbr':>9} {'stitch':>8} {'ft_gen':>8}   V3 strategy")
strategy = {}
for sub in sorted(SUBSET_TO_LANG):
    tag, a, m = choice.get(sub, ('2leg', 0.05, 99.0))
    pools = val_cands_all if tag == '2leg' else v4c
    idxs = [i for i in range(len(val_df)) if str(val_df.iloc[i]['subset']) == sub
            and pools[i] and str(val_df.iloc[i]['output']).strip()]
    hold = idxs[1::2]
    s_comb, s_st, s_gen = [], [], []
    g = uni_stitch_gate.get(sub)
    gv = gen_val.get(sub, {})
    for i in hold:
        rt = ref_t(i)
        dd, ddw, u1, uL = uni_prep(pools[i])
        s_comb.append(score_pair(rt, dd[mbr_idx(0.5*u1+0.5*uL, ddw, a, m)]))
        if g and g['use']:
            s_st.append(score_pair(rt, uni_stitch(pools[i], g['lam'], sub)) - STITCH_PENALTY*0)
        if str(i) in gv:
            s_gen.append(score_pair(rt, gv[str(i)]))
    opts = {'comb_mbr': np.mean(s_comb)}
    if s_st:  opts['stitch'] = np.mean(s_st) - 0.37*0  # rouge-only compare; penalty applied at pick
    if s_gen: opts['ft_gen'] = np.mean(s_gen)
    # pick: ft_gen/stitch must beat comb_mbr by margin (stitch additionally pays judge-risk penalty)
    pick = 'comb_mbr'
    if 'ft_gen' in opts and opts['ft_gen'] > opts['comb_mbr'] + 0.003: pick = 'ft_gen'
    if 'stitch' in opts and opts['stitch'] - STITCH_PENALTY > opts.get(pick, opts['comb_mbr']) + 0.003:
        pick = 'stitch'
    strategy[sub] = pick
    print(f"{sub:<10} {opts.get('comb_mbr', float('nan')):>9.4f} "
          f"{opts.get('stitch', float('nan')):>8.4f} {opts.get('ft_gen', float('nan')):>8.4f}   {pick}")
save = {k: v for k, v in strategy.items()}
json.dump(save, open(OUTPUT_DIR / 'v3_strategy.json', 'w'))

# ---- assemble V3 ----
ft_test_ans = json.load(open(OUTPUT_DIR / 'ftqwen_test_answers.json'))
rows_v3 = []
for i in tqdm(range(len(test_df)), desc="V3 assemble"):
    rid = str(test_df.iloc[i]['ID']); sub = test_subs[i]
    tag, a, m = choice.get(sub, ('2leg', 0.05, 99.0))
    if tag == '4leg':
        pool = union4(test_qs[i].strip(), test_emb[i], gem_test[i], bge_test[i], sub, exclude_exact=False)
    else:
        pool = get_same_lang_candidates(test_qs[i].strip(), test_emb[i], sub,
                                        k=K_CANDIDATES, exclude_exact=False)
    if not pool:
        rows_v3.append({'ID': test_df.iloc[i]['ID'], 'TargetR1F1': 'No answer',
                        'TargetRLF1': 'No answer', 'TargetLLM': 'No answer'}); continue
    pick = strategy.get(sub, 'comb_mbr')
    gen = ft_test_ans.get(rid, '')
    if pick == 'ft_gen' and gen:
        ans = gen
    elif pick == 'stitch':
        ans = uni_stitch(pool, uni_stitch_gate[sub]['lam'], sub)
    else:
        dd, ddw, u1, uL = uni_prep(pool)
        ans = dd[mbr_idx(0.5*u1+0.5*uL, ddw, a, m)]
    rows_v3.append({'ID': test_df.iloc[i]['ID'], 'TargetR1F1': ans,
                    'TargetRLF1': ans, 'TargetLLM': ans})

df3 = pd.DataFrame(rows_v3).reindex(columns=SUB_COLS)
assert len(df3) == len(test_df) and not df3.isnull().any().any()
df3.to_csv(OUTPUT_DIR / 'submission_v3_best_identical.csv', index=False)
print("\nSaved: submission_v3_best_identical.csv")
print("Strategy:", strategy)

Sub         comb_mbr   stitch   ft_gen   V3 strategy
Aka_Gha       0.2099   0.2182   0.2009   stitch
Amh_Eth       0.1467   0.1491      nan   comb_mbr
Eng_Eth       0.4950      nan      nan   comb_mbr
Eng_Gha       0.2087   0.2114   0.2715   ft_gen
Eng_Ken       0.6463      nan      nan   comb_mbr
Eng_Uga       0.6399      nan      nan   comb_mbr
Lug_Uga       0.6124      nan      nan   comb_mbr
Swa_Ken       0.6919      nan      nan   comb_mbr


V3 assemble: 100%|██████████| 2618/2618 [00:14<00:00, 178.36it/s]



Saved: submission_v3_best_identical.csv
Strategy: {'Aka_Gha': 'stitch', 'Amh_Eth': 'comb_mbr', 'Eng_Eth': 'comb_mbr', 'Eng_Gha': 'ft_gen', 'Eng_Ken': 'comb_mbr', 'Eng_Uga': 'comb_mbr', 'Lug_Uga': 'comb_mbr', 'Swa_Ken': 'comb_mbr'}


In [ ]:
ft_test_ans = json.load(open(OUTPUT_DIR/'ftqwen_test_answers.json'))
eg_val = json.load(open(OUTPUT_DIR/'ftqwen_enggha_val.json'))
tl = [len(uni_toks(v)) for k,v in ft_test_ans.items()
      if test_subs[[i for i in range(len(test_df)) if str(test_df.iloc[i]['ID'])==k][0]]=='Eng_Gha'] \
     if False else [len(uni_toks(ft_test_ans[str(test_df.iloc[i]['ID'])])) for i in range(len(test_df))
      if test_subs[i]=='Eng_Gha' and str(test_df.iloc[i]['ID']) in ft_test_ans]
vl = [len(uni_toks(v)) for v in eg_val.values()]
print(f"test gen len: median {np.median(tl):.0f}, short(<10): {sum(1 for x in tl if x<10)}")
print(f"val  gen len: median {np.median(vl):.0f}, short(<10): {sum(1 for x in vl if x<10)}")
# eyeball 3 test generations:
for i in [i for i in range(len(test_df)) if test_subs[i]=='Eng_Gha'][:3]:
    print('---', ft_test_ans[str(test_df.iloc[i]['ID'])][:200])

test gen len: median 69, short(<10): 0
val  gen len: median 50, short(<10): 1
--- Strategies for promoting awareness, education, and empowerment regarding sexual and reproductive health among migrant and mobile adolescents include: Community outreach programs that provide informati
--- Yes, school-based crisis intervention programs can accommodate diverse cultural or religious backgrounds by providing culturally sensitive and inclusive support. This may involve offering resources an
--- Platforms and social media companies play a crucial role in promoting consent and preventing sexual violence in online spaces by implementing policies and guidelines that prioritize user safety, priva


In [ ]:
# =============================================================================
# A: PER-LANGUAGE EMBEDDING INTERPOLATION  beta*AfriE5 + (1-beta)*FT2
# Harvests FT2's oracle gains without its top-1 forgetting. CPU only.
# =============================================================================
import numpy as np
from tqdm import tqdm

ft2_corpus = np.load(CACHE/'ft2_corpus.npy'); ft2_val = np.load(CACHE/'ft2_val.npy')
ft2_test = np.load(CACHE/'ft2_test.npy')
BETAS = [1.0, 0.9, 0.8, 0.7, 0.6, 0.5]

def interp_pool(i_val, sub, beta, k=K_CANDIDATES):
    """Top-k by interpolated similarity, same language."""
    _, mask = lang_indices[sub]
    mask = np.array(mask)
    s_old = corpus_emb[mask] @ val_emb[i_val]
    s_new = ft2_corpus[mask] @ ft2_val[i_val]
    s = beta*s_old + (1-beta)*s_new
    order = np.argsort(-s)
    out = []
    q = val_qs[i_val].strip()
    for j in order[:k+5]:
        ci = int(mask[j])
        if corpus_q_stripped[ci] == q: continue
        out.append({'answer': answers_raw[ci], 'sim': float(s[j]), 'idx': ci})
        if len(out) >= k: break
    return out

print(f"{'Sub':<10} {'beta':>5} {'cur wR':>8} {'interp wR':>9} {'delta':>8}  use")
interp_gate = {}
for sub in ['Eng_Uga', 'Lug_Uga', 'Eng_Eth', 'Eng_Ken', 'Swa_Ken']:   # strong langs only
    tag, a, m = choice.get(sub, ('2leg', 0.05, 99.0))
    idxs = [i for i in range(len(val_df)) if str(val_df.iloc[i]['subset']) == sub
            and val_cands_all[i] and str(val_df.iloc[i]['output']).strip()]
    tune, hold = idxs[0::2], idxs[1::2]
    def wscore(pool_fn, ix):
        tot = []
        for i in ix:
            pool = pool_fn(i)
            if not pool: continue
            dd, ddw, u1, uL = uni_prep(pool)
            ans = dd[mbr_idx(0.5*u1+0.5*uL, ddw, a, m)]
            rt = uni_toks(str(val_df.iloc[i]['output']))[:CAP]
            at = uni_toks(ans)[:CAP]
            tot.append(0.37*uni_r1(rt, at) + 0.37*uni_rl(rt, at))
        return np.mean(tot)
    cur_hold = wscore(lambda i: val_cands_all[i], hold)
    best_beta, best_tune = 1.0, -1
    for b in BETAS[1:]:
        s = wscore(lambda i: interp_pool(i, sub, b), tune[:300])   # 300 tune queries: fast
        if s > best_tune: best_tune, best_beta = s, b
    ih = wscore(lambda i: interp_pool(i, sub, best_beta), hold)
    use = ih > cur_hold + 0.002
    interp_gate[sub] = (use, best_beta)
    print(f"{sub:<10} {best_beta:>5.1f} {cur_hold:>8.4f} {ih:>9.4f} {ih-cur_hold:>+8.4f}  "
          f"{'INTERP' if use else 'keep'}")
import json; json.dump({k:[bool(v[0]),v[1]] for k,v in interp_gate.items()},
                       open(OUTPUT_DIR/'interp_gate.json','w'))
print("\nGate:", interp_gate)

Sub         beta   cur wR interp wR    delta  use
Eng_Uga      0.6   0.6399    0.6425  +0.0026  INTERP
Lug_Uga      0.6   0.6124    0.6174  +0.0050  INTERP
Eng_Eth      0.8   0.4950    0.4952  +0.0002  keep
Eng_Ken      0.8   0.6463    0.6410  -0.0053  keep
Swa_Ken      0.9   0.6919    0.6960  +0.0040  INTERP

Gate: {'Eng_Uga': (np.True_, 0.6), 'Lug_Uga': (np.True_, 0.6), 'Eng_Eth': (np.False_, 0.8), 'Eng_Ken': (np.False_, 0.8), 'Swa_Ken': (np.True_, 0.9)}


In [ ]:
# =============================================================================
# B: COMPLETENESS TIE-BREAK — among near-tied consensus candidates, prefer
# the most complete (longest). Adopt only if ROUGE-neutral on holdout.
# =============================================================================
TIE = 0.01
print(f"{'Sub':<10} {'cur wR':>8} {'tiebrk wR':>9} {'delta':>8} {'%swapped':>9}  use")
tie_gate = {}
for sub in sorted(SUBSET_TO_LANG):
    tag, a, m = choice.get(sub, ('2leg', 0.05, 99.0))
    pools = val_cands_all if tag == '2leg' else v4c
    idxs = [i for i in range(len(val_df)) if str(val_df.iloc[i]['subset'])==sub
            and pools[i] and str(val_df.iloc[i]['output']).strip()]
    hold = idxs[1::2]
    cur, tb, swapped = [], [], 0
    for i in hold:
        dd, ddw, u1, uL = uni_prep(pools[i])
        util = 0.5*u1 + 0.5*uL
        base_ix = mbr_idx(util, ddw, a, m)
        # tie set: within TIE of the chosen candidate's utility
        ties = [j for j in range(len(dd)) if util[base_ix] - util[j] <= TIE]
        long_ix = max(ties, key=lambda j: len(uni_toks(dd[j])))
        if long_ix != base_ix: swapped += 1
        rt = uni_toks(str(val_df.iloc[i]['output']))[:CAP]
        for lst, ix in ((cur, base_ix), (tb, long_ix)):
            at = uni_toks(dd[ix])[:CAP]
            lst.append(0.37*uni_r1(rt, at) + 0.37*uni_rl(rt, at))
    d = np.mean(tb) - np.mean(cur)
    use = d > -0.001            # ROUGE-neutral or better -> judge upside is free
    tie_gate[sub] = use
    print(f"{sub:<10} {np.mean(cur):>8.4f} {np.mean(tb):>9.4f} {d:>+8.4f} "
          f"{100*swapped/len(hold):>8.1f}%  {'TIEBRK' if use else 'keep'}")
print("\nGate:", tie_gate)
print("Note: ROUGE measures the cost; the judge benefit is only measurable on the LB.")

Sub          cur wR tiebrk wR    delta  %swapped  use
Aka_Gha      0.2099    0.2037  -0.0062     42.0%  keep
Amh_Eth      0.1467    0.1151  -0.0316     85.3%  keep
Eng_Eth      0.4950    0.3469  -0.1481     71.6%  keep
Eng_Gha      0.2087    0.1880  -0.0207     72.8%  keep
Eng_Ken      0.6463    0.2402  -0.4061     86.2%  keep
Eng_Uga      0.6399    0.4950  -0.1449     41.0%  keep
Lug_Uga      0.6124    0.2433  -0.3690     79.9%  keep
Swa_Ken      0.6919    0.2513  -0.4407     84.9%  keep

Gate: {'Aka_Gha': np.False_, 'Amh_Eth': np.False_, 'Eng_Eth': np.False_, 'Eng_Gha': np.False_, 'Eng_Ken': np.False_, 'Eng_Uga': np.False_, 'Lug_Uga': np.False_, 'Swa_Ken': np.False_}
Note: ROUGE measures the cost; the judge benefit is only measurable on the LB.


In [ ]:
# ==== RL FULL-LENGTH RESCORING: top-5 shortlist re-ranked by untruncated LCS ====
print(f"{'Sub':<10} {'cur RL':>8} {'fullLCS RL':>10} {'delta':>8}  use")
full_gate = {}
for sub in sorted(SUBSET_TO_LANG):
    tag, a, m = choice.get(sub, ('2leg', 0.05, 99.0))
    pools = val_cands_all if tag == '2leg' else v4c
    idxs = [i for i in range(len(val_df)) if str(val_df.iloc[i]['subset'])==sub
            and pools[i] and str(val_df.iloc[i]['output']).strip()]
    hold = idxs[1::2]
    cur, full = [], []
    for i in hold[:400]:
        dd, ddw, u1, uL = uni_prep(pools[i])
        base_ix = mbr_idx(uL, ddw, a, m)
        short = list(np.argsort(-uL))[:5]
        toks_full = [uni_toks(dd[j])[:CAP] for j in short]
        # full-length pairwise LCS consensus among shortlist, weighted
        best_j, best_u = short[0], -1
        for jj, j in enumerate(short):
            u = sum(ddw[short[kk]] * uni_rl(toks_full[jj], toks_full[kk])
                    for kk in range(len(short)) if kk != jj)
            if u > best_u: best_u, best_j = u, j
        rt = uni_toks(str(val_df.iloc[i]['output']))[:CAP]
        cur.append(uni_rl(rt, uni_toks(dd[base_ix])[:CAP]))
        full.append(uni_rl(rt, uni_toks(dd[best_j])[:CAP]))
    d = np.mean(full) - np.mean(cur)
    full_gate[sub] = d > 0.002
    print(f"{sub:<10} {np.mean(cur):>8.4f} {np.mean(full):>10.4f} {d:>+8.4f}  "
          f"{'FULL' if full_gate[sub] else 'keep'}")
print("Gate:", full_gate)

Sub          cur RL fullLCS RL    delta  use
Aka_Gha      0.2085     0.2036  -0.0048  keep
Amh_Eth      0.1912     0.1663  -0.0249  keep
Eng_Eth      0.6592     0.4499  -0.2093  keep
Eng_Gha      0.2156     0.2014  -0.0142  keep
Eng_Ken      0.8678     0.2348  -0.6330  keep
Eng_Uga      0.8553     0.5691  -0.2862  keep
Lug_Uga      0.8223     0.1895  -0.6328  keep
Swa_Ken      0.9297     0.2297  -0.7000  keep
Gate: {'Aka_Gha': np.False_, 'Amh_Eth': np.False_, 'Eng_Eth': np.False_, 'Eng_Gha': np.False_, 'Eng_Ken': np.False_, 'Eng_Uga': np.False_, 'Lug_Uga': np.False_, 'Swa_Ken': np.False_}


In [ ]:
# =============================================================================
# V4: V3 winners + embedding interpolation (Eng_Uga/Lug_Uga/Swa_Ken)
#     + Aka_Gha re-decided on full weighted holdout. Identical columns, open-source.
# =============================================================================
import json
import numpy as np, pandas as pd
from tqdm import tqdm

ft2_corpus = np.load(CACHE/'ft2_corpus.npy'); ft2_test = np.load(CACHE/'ft2_test.npy')
INTERP = {'Eng_Uga': 0.6, 'Lug_Uga': 0.6, 'Swa_Ken': 0.9}   # adopted betas

def interp_pool_test(i, sub, beta, k=K_CANDIDATES):
    _, mask = lang_indices[sub]
    mask = np.array(mask)
    s = beta*(corpus_emb[mask] @ test_emb[i]) + (1-beta)*(ft2_corpus[mask] @ ft2_test[i])
    order = np.argsort(-s)
    out = []
    for j in order[:k+5]:
        ci = int(mask[j])
        out.append({'answer': answers_raw[ci], 'sim': float(s[j]), 'idx': ci})
        if len(out) >= k: break
    return out   # NOTE: exclude_exact=False behavior — duplicates kept (test time)

# ---- Aka_Gha re-decision: stitch vs comb_mbr on FULL weighted holdout ----
sub = 'Aka_Gha'; tag, a, m = choice[sub]
idxs = [i for i in range(len(val_df)) if str(val_df.iloc[i]['subset'])==sub
        and val_cands_all[i] and str(val_df.iloc[i]['output']).strip()]
hold = idxs[1::2]
s_st, s_cb = [], []
for i in hold:
    rt = uni_toks(str(val_df.iloc[i]['output']))[:CAP]
    st = uni_toks(uni_stitch(val_cands_all[i], 0.70, sub))[:CAP]
    dd, ddw, u1, uL = uni_prep(val_cands_all[i])
    cb = uni_toks(dd[mbr_idx(0.5*u1+0.5*uL, ddw, a, m)])[:CAP]
    s_st.append(0.37*uni_r1(rt, st) + 0.37*uni_rl(rt, st))
    s_cb.append(0.37*uni_r1(rt, cb) + 0.37*uni_rl(rt, cb))
AKA_PICK = 'stitch' if np.mean(s_st) > np.mean(s_cb) + 0.002 else 'comb_mbr'
print(f"Aka_Gha full-weighted holdout: stitch={np.mean(s_st):.4f} comb={np.mean(s_cb):.4f} -> {AKA_PICK}")

strategy = json.load(open(OUTPUT_DIR/'v3_strategy.json'))
strategy['Aka_Gha'] = AKA_PICK
print("V4 strategy:", strategy)

# ---- assemble ----
ft_test_ans = json.load(open(OUTPUT_DIR/'ftqwen_test_answers.json'))
rows_v4 = []
for i in tqdm(range(len(test_df)), desc="V4 assemble"):
    rid = str(test_df.iloc[i]['ID']); sub = test_subs[i]
    tag, a, m = choice.get(sub, ('2leg', 0.05, 99.0))
    if sub in INTERP:
        pool = interp_pool_test(i, sub, INTERP[sub])
    elif tag == '4leg':
        pool = union4(test_qs[i].strip(), test_emb[i], gem_test[i], bge_test[i], sub, exclude_exact=False)
    else:
        pool = get_same_lang_candidates(test_qs[i].strip(), test_emb[i], sub,
                                        k=K_CANDIDATES, exclude_exact=False)
    if not pool:
        rows_v4.append({'ID': test_df.iloc[i]['ID'], 'TargetR1F1': 'No answer',
                        'TargetRLF1': 'No answer', 'TargetLLM': 'No answer'}); continue
    pick = strategy.get(sub, 'comb_mbr')
    gen = ft_test_ans.get(rid, '')
    if pick == 'ft_gen' and gen:
        ans = gen
    elif pick == 'stitch':
        ans = uni_stitch(pool, uni_stitch_gate[sub]['lam'], sub)
    else:
        dd, ddw, u1, uL = uni_prep(pool)
        ans = dd[mbr_idx(0.5*u1+0.5*uL, ddw, a, m)]
    rows_v4.append({'ID': test_df.iloc[i]['ID'], 'TargetR1F1': ans,
                    'TargetRLF1': ans, 'TargetLLM': ans})

df4 = pd.DataFrame(rows_v4).reindex(columns=SUB_COLS)
assert len(df4) == len(test_df) and not df4.isnull().any().any()
df4.to_csv(OUTPUT_DIR/'submission_v4_final.csv', index=False)
print("Saved: submission_v4_final.csv")

Aka_Gha full-weighted holdout: stitch=0.2182 comb=0.2099 -> stitch
V4 strategy: {'Aka_Gha': 'stitch', 'Amh_Eth': 'comb_mbr', 'Eng_Eth': 'comb_mbr', 'Eng_Gha': 'ft_gen', 'Eng_Ken': 'comb_mbr', 'Eng_Uga': 'comb_mbr', 'Lug_Uga': 'comb_mbr', 'Swa_Ken': 'comb_mbr'}


V4 assemble: 100%|██████████| 2618/2618 [00:35<00:00, 74.55it/s]


Saved: submission_v4_final.csv


In [ ]:
# =============================================================================
# AFROLM SEMANTIC CHECK: phase-2 metric proxy on val holdout, per V4 strategy
# =============================================================================
import torch, json
import numpy as np
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm

AFRO = 'bonadossou/afrolm_active_learning'
atok = AutoTokenizer.from_pretrained(AFRO)
amod = AutoModel.from_pretrained(AFRO).to('cuda:0' if torch.cuda.is_available() else 'cpu').eval()
DEV = next(amod.parameters()).device

@torch.no_grad()
def afro_embed(texts, bs=32):
    out = []
    for s in range(0, len(texts), bs):
        enc = atok(texts[s:s+bs], padding=True, truncation=True, max_length=256,
                   return_tensors='pt').to(DEV)
        h = amod(**enc).last_hidden_state
        mask = enc['attention_mask'].unsqueeze(-1)
        emb = (h*mask).sum(1) / mask.sum(1)
        out.append(torch.nn.functional.normalize(emb, dim=-1).cpu())
    return torch.cat(out)

strategy = json.load(open(OUTPUT_DIR/'v3_strategy.json'));
eg = json.load(open(OUTPUT_DIR/'ftqwen_enggha_val.json'))
print(f"{'Sub':<10} {'strategy':>9} {'AfroLM cos':>11} {'top1 cos':>9}  verdict")
for sub in sorted(SUBSET_TO_LANG):
    tag, a, m = choice.get(sub, ('2leg', 0.05, 99.0))
    pools = val_cands_all if tag == '2leg' else v4c
    idxs = [i for i in range(len(val_df)) if str(val_df.iloc[i]['subset'])==sub
            and pools[i] and str(val_df.iloc[i]['output']).strip()][:200]
    pick = strategy.get(sub, 'comb_mbr')
    refs, anss, t1s = [], [], []
    for i in idxs:
        refs.append(str(val_df.iloc[i]['output']).strip())
        t1s.append(pools[i][0]['answer'])
        if pick == 'ft_gen' and str(i) in eg: anss.append(eg[str(i)])
        elif pick == 'stitch': anss.append(uni_stitch(pools[i], uni_stitch_gate[sub]['lam'], sub))
        else:
            dd, ddw, u1, uL = uni_prep(pools[i])
            anss.append(dd[mbr_idx(0.5*u1+0.5*uL, ddw, a, m)])
    R, A, T = afro_embed(refs), afro_embed(anss), afro_embed(t1s)
    cos_a = float((R*A).sum(-1).mean()); cos_t = float((R*T).sum(-1).mean())
    print(f"{sub:<10} {pick:>9} {cos_a:>11.4f} {cos_t:>9.4f}  "
          f"{'OK' if cos_a >= cos_t - 0.01 else 'REVIEW'}")

config.json:   0%|          | 0.00/755 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/241 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/6.01M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.06G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/165 [00:00<?, ?it/s]

[transformers] XLMRobertaModel LOAD REPORT from: bonadossou/afrolm_active_learning
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Sub         strategy  AfroLM cos  top1 cos  verdict


model.safetensors:   0%|          | 0.00/1.06G [00:00<?, ?B/s]

Aka_Gha       stitch      0.8803    0.8737  OK
Amh_Eth     comb_mbr      0.8221    0.8221  OK
Eng_Eth     comb_mbr      0.9632    0.9632  OK
Eng_Gha       ft_gen      0.9412    0.9414  OK
Eng_Ken     comb_mbr      0.9842    0.9842  OK
Eng_Uga     comb_mbr      0.9813    0.9813  OK
Lug_Uga     comb_mbr      0.9849    0.9849  OK
Swa_Ken     comb_mbr      0.9947    0.9947  OK


In [ ]:
import torch
@torch.no_grad()
def ft_generate(q, lang, cands, max_new=350):
    ctx = "\n".join(f"{k+1}. {c['answer']}" for k, c in enumerate(cands[:5]))
    msgs = [{"role":"system","content":
             f"You are a multilingual health expert. Answer health questions based on the "
             f"reference information provided. Use the EXACT words and phrases from the "
             f"references when possible. Be complete and accurate. Answer in {lang}."},
            {"role":"user","content": f"Question: {q}\n\nReference answers:\n{ctx}"}]
    enc = tok.apply_chat_template(msgs, add_generation_prompt=True,
                                  return_tensors='pt', return_dict=True).to('cuda:0')
    out = ft.generate(**enc, max_new_tokens=max_new, do_sample=False,
                      pad_token_id=tok.eos_token_id)
    return tok.decode(out[0][enc['input_ids'].shape[1]:], skip_special_tokens=True).strip()

In [ ]:
# ==== LUG_UGA FULL-VAL GENERATION PROBE (resumable) ====
# Requires: bootstrap + ft/tok loaded (bf16 cell)
import json
lg_idxs = [i for i in range(len(val_df)) if str(val_df.iloc[i]['subset'])=='Lug_Uga'
           and val_cands_all[i] and str(val_df.iloc[i]['output']).strip()]
lgp = OUTPUT_DIR / 'ftqwen_luguga_val.json'
lg = json.load(open(lgp)) if lgp.exists() else {}
for n, i in enumerate(tqdm(lg_idxs, desc="Gen Lug_Uga val")):
    if str(i) in lg: continue
    lg[str(i)] = ft_generate(val_qs[i], 'Luganda (Uganda)', val_cands_all[i])
    if (n+1) % 50 == 0: json.dump(lg, open(lgp, 'w'))
json.dump(lg, open(lgp, 'w'))

hold = lg_idxs[1::2]
tag, a, m = choice['Lug_Uga']
g_w, c_w = [], []
for i in hold:
    rt = uni_toks(str(val_df.iloc[i]['output']))[:CAP]
    gt = uni_toks(lg[str(i)])[:CAP]
    dd, ddw, u1, uL = uni_prep(val_cands_all[i])
    ct = uni_toks(dd[mbr_idx(0.5*u1+0.5*uL, ddw, a, m)])[:CAP]
    g_w.append(0.37*uni_r1(rt, gt) + 0.37*uni_rl(rt, gt))
    c_w.append(0.37*uni_r1(rt, ct) + 0.37*uni_rl(rt, ct))
print(f"Lug_Uga holdout (n={len(hold)}): gen={np.mean(g_w):.4f}  comb_mbr={np.mean(c_w):.4f}  "
      f"Δ={np.mean(g_w)-np.mean(c_w):+.4f}")
print("ADOPT if Δ > +0.005 (strong-lang transfer ~1:1, but gen is new territory here — margin matters).")

Gen Lug_Uga val: 100%|██████████| 846/846 [4:19:14<00:00, 18.39s/it]


Lug_Uga holdout (n=423): gen=0.5231  comb_mbr=0.6124  Δ=-0.0892
ADOPT if Δ > +0.005 (strong-lang transfer ~1:1, but gen is new territory here — margin matters).


In [25]:
# ==== INTERPOLATION ROUND 2: finer+wider beta grid, all 8 languages ====
import numpy as np, json
from tqdm import tqdm
ft2_corpus = np.load(CACHE/'ft2_corpus.npy'); ft2_val = np.load(CACHE/'ft2_val.npy')
BETAS = [1.0, 0.8, 0.7, 0.6, 0.5, 0.4, 0.3, 0.2]

def interp_pool(i_val, sub, beta, k=K_CANDIDATES):
    _, mask = lang_indices[sub]; mask = np.array(mask)
    s = beta*(corpus_emb[mask] @ val_emb[i_val]) + (1-beta)*(ft2_corpus[mask] @ ft2_val[i_val])
    order = np.argsort(-s); out = []
    q = val_qs[i_val].strip()
    for j in order[:k+5]:
        ci = int(mask[j])
        if corpus_q_stripped[ci] == q: continue
        out.append({'answer': answers_raw[ci], 'sim': float(s[j]), 'idx': ci})
        if len(out) >= k: break
    return out

print(f"{'Sub':<10} {'beta':>5} {'cur wR':>8} {'interp wR':>9} {'delta':>8}  use")
interp2 = {}
for sub in sorted(SUBSET_TO_LANG):
    tag, a, m = choice.get(sub, ('2leg', 0.05, 99.0))
    idxs = [i for i in range(len(val_df)) if str(val_df.iloc[i]['subset'])==sub
            and val_cands_all[i] and str(val_df.iloc[i]['output']).strip()]
    tune, hold = idxs[0::2], idxs[1::2]
    def wsc(pool_fn, ix):
        tot = []
        for i in ix:
            pool = pool_fn(i)
            if not pool: continue
            dd, ddw, u1, uL = uni_prep(pool)
            ans = dd[mbr_idx(0.5*u1+0.5*uL, ddw, a, m)]
            rt = uni_toks(str(val_df.iloc[i]['output']))[:CAP]; at = uni_toks(ans)[:CAP]
            tot.append(0.37*uni_r1(rt, at) + 0.37*uni_rl(rt, at))
        return np.mean(tot)
    cur_h = wsc(lambda i: val_cands_all[i], hold)
    bb, bt = 1.0, -1
    for b in BETAS[1:]:
        s = wsc(lambda i: interp_pool(i, sub, b), tune[:300])
        if s > bt: bt, bb = s, b
    ih = wsc(lambda i: interp_pool(i, sub, bb), hold)
    use = ih > cur_h + 0.002
    interp2[sub] = (bool(use), bb)
    print(f"{sub:<10} {bb:>5.1f} {cur_h:>8.4f} {ih:>9.4f} {ih-cur_h:>+8.4f}  {'INTERP' if use else 'keep'}")
json.dump({k: list(v) for k, v in interp2.items()}, open(OUTPUT_DIR/'interp_gate2.json','w'))
print("Gate:", interp2)

Sub         beta   cur wR interp wR    delta  use
Aka_Gha      0.3   0.2099    0.2098  -0.0002  keep
Amh_Eth      0.8   0.1467    0.1468  +0.0000  keep
Eng_Eth      0.8   0.4950    0.4952  +0.0002  keep
Eng_Gha      0.2   0.2020    0.2169  +0.0149  INTERP
Eng_Ken      0.8   0.6463    0.6410  -0.0053  keep
Eng_Uga      0.3   0.6399    0.6418  +0.0019  keep
Lug_Uga      0.6   0.6124    0.6174  +0.0050  INTERP
Swa_Ken      0.8   0.6919    0.6980  +0.0061  INTERP
Gate: {'Aka_Gha': (False, 0.3), 'Amh_Eth': (False, 0.8), 'Eng_Eth': (False, 0.8), 'Eng_Gha': (True, 0.2), 'Eng_Ken': (False, 0.8), 'Eng_Uga': (False, 0.3), 'Lug_Uga': (True, 0.6), 'Swa_Ken': (True, 0.8)}
